# Halden Energy — Quarter 7: Pelican cuts prices and announces a refinery
**Python notebook (the same data as the workbook)**

Pelican Energy cut what drivers pay by 6 cents a gallon near Cordell stations, in every kind of market, and announced a new refining unit on the Gulf Coast.

Two different problems. At the stations, it is local price competition: match or hold, market by market. At the refinery, it is a game of commitment, where the announcement itself may be the move.

Unzip the data files next to this notebook before you run it.

In [ ]:
import pandas as pd

markets = pd.read_csv('data/cordell_markets.csv')
pelican = pd.read_csv('data/pelican_cut.csv').set_index('parameter')['value']
game = pd.read_csv('data/refinery_game.csv')
shop = pd.read_csv('data/shop_margin_response.csv').set_index('parameter')['value']
examples = pd.read_csv('data/industry_examples.csv')
markets

## Part 1 — the worked example: two stations, one cut

A rival cuts 10 cents. A corner station with three rivals loses drivers on pennies (elasticity −20). The only station for twenty miles barely notices (−0.03). Same cut, opposite answers.

In [ ]:
ex = pd.read_csv('data/worked_example.csv').set_index('parameter')['value']
for name, e in [('corner station', ex['station_elasticity_switching']), ('lonely station', ex['station_elasticity_loyal'])]:
    at_risk = e * ex['cut_usd_gal'] / ex['pump_usd_gal']          # share of volume that leaves if you hold
    hold_cost = ex['volume_m_gal'] * ex['margin_usd_gal'] * abs(at_risk)
    match_cost = ex['volume_m_gal'] * ex['cut_usd_gal']
    print(f'{name:14s} hold costs ${hold_cost:7.2f}M a year, match costs ${match_cost:6.1f}M a year')

## Part 2 — Cordell's four markets

For each market: what share of volume leaves if you hold, what those gallons were worth, and what matching gives up. A gallon is worth Cordell's fuel margin plus Halden's share of the dealer's shop margin.

In [ ]:
value_per_gal = pelican['cordell_fuel_margin_usd_gal'] + pelican['shop_margin_usd_gal'] * pelican['halden_share_of_shop_margin']
gap = pelican['street_cut_usd_gal'] / pelican['pump_price_usd_gal']
out = markets.copy()
out['volume_at_risk_pct'] = out['elasticity'] * gap * 100
out['hold_cost_musd_qtr'] = out['gallons_per_quarter_m'] * value_per_gal * out['volume_at_risk_pct'].abs() / 100
out['match_cost_musd_qtr'] = out['gallons_per_quarter_m'] * pelican['street_cut_usd_gal']
out[['label', 'elasticity', 'gallons_per_quarter_m', 'volume_at_risk_pct', 'hold_cost_musd_qtr', 'match_cost_musd_qtr']].round(3)

## Part 3 — the refinery game

Ana's team put the chance Pelican really builds at 70%. Weigh each answer by that chance, then ask: at what chance would the answer flip?

In [ ]:
p = pelican['chance_pelican_builds']
pay = {(r.halden, r.pelican): r.halden_payoff_musd_per_year for r in game.itertuples()}
ev_hold = p * pay[('hold', 'builds')] + (1 - p) * pay[('hold', 'bluffs')]
ev_build = p * pay[('match', 'builds')] + (1 - p) * pay[('match', 'bluffs')]
flip = (pay[('match', 'bluffs')] - pay[('hold', 'bluffs')]) / ((pay[('match', 'bluffs')] - pay[('hold', 'bluffs')]) + (pay[('hold', 'builds')] - pay[('match', 'builds')]))
print(f'Expected payoff, hold:  ${ev_hold:6.1f}M a year')
print(f'Expected payoff, build: ${ev_build:6.1f}M a year')
print(f'The answers are worth the same when the chance Pelican builds is {flip:.1%}')

## Part 4 — what the whole industry does comes back as the shop margin

If most chains match, drivers learn to shop on price and stop buying coffee. The shop margin two quarters from now moves with the industry's combined aggression this quarter.

In [ ]:
def shop_margin(aggression):
    v = shop['base_shop_margin_usd_gal'] - shop['slope_usd_gal'] * (aggression - shop['pivot'])
    lo, hi = shop['base_shop_margin_usd_gal'] * (1 - shop['bound']), shop['base_shop_margin_usd_gal'] * (1 + shop['bound'])
    return min(hi, max(lo, v))

for r in examples.itertuples():
    print(f'{r.example:12s} aggression {r.aggression:.1f} -> shop margin ${shop_margin(r.aggression):.3f} a gallon')

## Your analysis

Fill in the `# TODO` cells.

In [ ]:
# TODO: in which markets, if any, does matching cost less than holding?
# TODO: what would the elasticity in the cities have to be for matching there to pay?
# TODO: hold or build at the refinery? What do you believe about Pelican, and how wrong can you afford to be?
# TODO: if every team in your class matched everywhere, what happens to your shop profit per station two quarters from now?

## Your decision

- Market by market: hold or match?
- The refinery: hold or build too?
- Which of these two answers depends on arithmetic, and which depends on what you believe Pelican will do?